# Module 2: Emotion Classification

This notebook fine-tunes a transformer emotion classifier for the mental health chatbot. It is designed for Google Colab with a T4 GPU, while the project repo keeps the reusable inference and explanation code locally.

## Why DistilBERT?

We use `distilbert-base-uncased` because it is transformer-based, lighter than BERT, fast enough for Colab T4 training, and practical for local inference. For this project, that balance is better than a larger model that may be harder to deploy.

Dataset: `dair-ai/emotion` with six labels: sadness, joy, love, anger, fear, and surprise.

In [ ]:
!nvidia-smi

In [ ]:
%pip -q install -U "transformers>=4.40,<5" "datasets>=2.18,<5" "accelerate>=0.28" "evaluate>=0.4" "scikit-learn>=1.4"

## Colab Repo Setup

Run this notebook from the cloned project folder so the trained model and reports are saved back into the same structure used locally.

In [ ]:
# If you opened the notebook outside the repo in Colab, uncomment these lines once:
# !git clone https://github.com/MarwanZaineldeen/Mental-Health-Chatbot.git
# %cd Mental-Health-Chatbot

In [ ]:
import json
from inspect import signature
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
MODEL_DIR = PROJECT_ROOT / 'src' / 'models' / 'saved_emotion_model'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'module_2_emotion_classification'
MODEL_NAME = 'distilbert-base-uncased'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
dataset = load_dataset('dair-ai/emotion', 'split')
label_names = dataset['train'].features['label'].names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

print(dataset)
print(id2label)
pd.DataFrame(dataset['train'][:5])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

encoded = dataset.map(tokenize, batched=True)
encoded = encoded.rename_column('label', 'labels')
encoded = encoded.remove_columns(['text'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, average='macro'),
    }

training_kwargs = {
    'output_dir': str(PROJECT_ROOT / 'checkpoints' / 'emotion_distilbert'),
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 16,
    'per_device_eval_batch_size': 32,
    'num_train_epochs': 5,
    'weight_decay': 0.01,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'macro_f1',
    'greater_is_better': True,
    'logging_steps': 50,
    'report_to': 'none',
}

strategy_name = 'eval_strategy' if 'eval_strategy' in signature(TrainingArguments).parameters else 'evaluation_strategy'
training_kwargs[strategy_name] = 'epoch'

args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    'model': model,
    'args': args,
    'train_dataset': encoded['train'],
    'eval_dataset': encoded['validation'],
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}

tokenizer_arg = 'processing_class' if 'processing_class' in signature(Trainer).parameters else 'tokenizer'
trainer_kwargs[tokenizer_arg] = tokenizer

trainer = Trainer(**trainer_kwargs)

In [ ]:
trainer.train()
validation_metrics = trainer.evaluate(encoded['validation'])
test_output = trainer.predict(encoded['test'])

test_predictions = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids
test_metrics = compute_metrics((test_output.predictions, test_labels))

print(validation_metrics)
print(test_metrics)

In [ ]:
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

all_label_ids = list(range(len(label_names)))
report_text = classification_report(test_labels, test_predictions, labels=all_label_ids, target_names=label_names, zero_division=0)
report_dict = classification_report(test_labels, test_predictions, labels=all_label_ids, target_names=label_names, output_dict=True, zero_division=0)
matrix = confusion_matrix(test_labels, test_predictions, labels=all_label_ids)

(REPORT_DIR / 'test_classification_report.txt').write_text(report_text, encoding='utf-8')
pd.DataFrame(report_dict).transpose().to_csv(REPORT_DIR / 'test_classification_report.csv')
pd.DataFrame(matrix, index=label_names, columns=label_names).to_csv(REPORT_DIR / 'test_confusion_matrix.csv')

summary = {
    'base_model': MODEL_NAME,
    'dataset': 'dair-ai/emotion',
    'labels': label_names,
    'validation_metrics': validation_metrics,
    'test_metrics': test_metrics,
    'explainability_method': 'word occlusion on final predicted confidence',
}
(REPORT_DIR / 'metrics_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(report_text)
print(f'Saved model to: {MODEL_DIR}')
print(f'Saved reports to: {REPORT_DIR}')

## Explainability Check

This is not a formal clinical explanation. It is a practical debugging tool: remove one word at a time and measure how much the model's confidence in the predicted emotion drops.

In [ ]:
def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    best_id = int(probabilities.argmax().item())
    return {
        'emotion': id2label[best_id],
        'confidence': float(probabilities[best_id].item()),
    }

def explain_by_word_removal(text, top_k=6):
    words = text.split()
    base_prediction = predict_emotion(text)
    base_emotion = base_prediction['emotion']
    base_confidence = base_prediction['confidence']
    evidence = []

    for index, word in enumerate(words):
        shortened_text = " ".join(words[:index] + words[index + 1:])
        if not shortened_text:
            continue

        shortened_prediction = predict_emotion(shortened_text)
        shortened_confidence = shortened_prediction["confidence"] if shortened_prediction["emotion"] == base_emotion else 0.0
        evidence.append({
            'word': word,
            'impact': round(base_confidence - shortened_confidence, 4),
        })

    evidence = sorted(evidence, key=lambda item: item['impact'], reverse=True)
    return {
        'text': text,
        'prediction': base_prediction,
        'top_evidence': evidence[:top_k],
    }

examples = [
    'I feel anxious and overwhelmed and I cannot sleep.',
    'I finally feel hopeful and proud of myself.',
    'I am angry because nobody listens to me.',
]

explanations = [explain_by_word_removal(text) for text in examples]
(REPORT_DIR / 'explanation_examples.json').write_text(json.dumps(explanations, indent=2), encoding='utf-8')
explanations